# NFL ATS Weekly Prediction Pipeline

Run via **papermill** from the project root:
```bash
papermill betting/predict_betting.ipynb /tmp/out.ipynb -p MODE thursday
```
**Modes:** `tuesday` · `thursday` · `sunday` · `backfill`

Or run all cells interactively — set `MODE` (and optionally `TARGET_WEEK`) in the Parameters cell.


## Parameters

Set `MODE` before running. Papermill overrides this parameter automatically:

| Mode | When | What it does |
|------|------|--------------|
| `tuesday` | Tue 9am ET | Updates previous week results, then runs new predictions |
| `thursday` | Thu 9pm ET | Refreshes predictions with latest injury reports |
| `sunday` | Sun 9am ET | Final predictions before kickoff |
| `backfill` | Manual | Runs predictions for `TARGET_WEEK`, then immediately fills in actual results |

`TARGET_WEEK` — override week auto-detection (required for `backfill`; optional for other modes).


In [1]:
MODE        = "thursday"  # tuesday | thursday | sunday | backfill
TARGET_WEEK = None         # set to an int to override auto-detection (backfill)    
TARGET_SEASON = None       # None = auto-detect from current date; override to lock a season


## Imports

All third-party and standard-library imports. `FinalCfg` — the training configuration dataclass — must be importable when `joblib.load()` deserialises the XGBoost pkl. It is defined in the **Paths & Configuration** cell below, which must run before any model-loading cell.

In [2]:
# Polars >= 1.x is strict about UTF-8 in parquet files; nflverse data
# occasionally contains non-UTF-8 strings. Fall back to pyarrow on failure.
import polars as _pl
_pl_read_parquet_orig = _pl.read_parquet
def _pl_read_parquet_lenient(source, *args, **kwargs):
    try:
        return _pl_read_parquet_orig(source, *args, **kwargs)
    except Exception:
        kwargs.setdefault('use_pyarrow', True)
        return _pl_read_parquet_orig(source, *args, **kwargs)
_pl.read_parquet = _pl_read_parquet_lenient

import os
import joblib
import lightgbm
import re as _re
import unicodedata as _ud
import numpy as np
import pandas as pd
import nflreadpy as nfl
from datetime import datetime
from dataclasses import dataclass, field
from typing import Tuple, Dict, Any
from pathlib import Path


In [3]:
# Smoke-test: verify all critical modules are in the namespace
assert 'pd'      in dir(), 'pandas missing'
assert 'np'      in dir(), 'numpy missing'
assert 'nfl'     in dir(), 'nflreadpy (nfl) not imported'
assert 'joblib'  in dir(), 'joblib missing'
assert 'datetime' in dir(), 'datetime missing'
print('✓ All imports present')

✓ All imports present


## Paths & Configuration

Resolves all file paths relative to the working directory — handles running from the project root (`BettingEdgeContinued/`) or from inside `betting/` directly. `FinalCfg` is also defined here: it must exist in the Python namespace *before* `joblib.load()` is called on the XGBoost pkl, because the dataclass is embedded in the serialised model and Python needs it at deserialisation time.

In [4]:
# ── Config ────────────────────────────────────────────────────────────────────
# Works whether kernel starts from project root or from betting/ directly
_cwd = Path.cwd()
_DIR = _cwd if _cwd.name == "betting" else _cwd / "betting"
_MODELS_DIR = _DIR / "models"

TRACKER_PATH    = str(_DIR / "predictions_tracker.csv")
ALLPRO_CSV_PATH = str(_DIR / "nfl_allpro_1997_2025.csv")
XGB_MODEL_PATH  = str(_MODELS_DIR / "xgboost_prod_model.pkl")
ENS_MODEL_PATH  = str(_MODELS_DIR / "ensemble_prod_model.pkl")
LGBM_MODEL_PATH = str(_MODELS_DIR / "lgbm_prod_model.pkl")

print(f"Running in {MODE.upper()} mode — {datetime.now().strftime('%Y-%m-%d %H:%M')}")

# ── Load XGBoost production model ─────────────────────────────────────────────
@dataclass(frozen=True)
class FinalCfg:
    test_size: float = 0.2
    random_state: int = 42
    oof_splits: int = 5
    weight_win: float = 2.0
    weight_loss: float = 1.0
    drop_non_features: Tuple[str, ...] = ('game_id', 'home_team', 'away_team', 'season', 'week')
    categorical_cols: Tuple[str, ...] = ('roof', 'surface')
    boolean_cols: Tuple[str, ...] = (
        'is_playoff', 'is_final_week', 'home_qb_switch', 'away_qb_switch',
        'is_home_qb_new', 'is_away_qb_new'
    )
    base_xgb_params: Dict[str, Any] = field(default_factory=lambda: dict(
        n_estimators=500, max_depth=3, learning_rate=0.01, min_child_weight=3,
        subsample=0.6, colsample_bytree=0.6, reg_alpha=2.0, reg_lambda=5.0,
        objective='reg:squarederror', random_state=42, tree_method='hist', n_jobs=1
    ))

Running in THURSDAY mode — 2026-05-19 20:42


In [5]:
from pathlib import Path as _P
_missing = [p for p in [XGB_MODEL_PATH, ENS_MODEL_PATH, LGBM_MODEL_PATH, ALLPRO_CSV_PATH]
            if not _P(p).exists()]
assert not _missing, f'Missing files (run from project root or betting/): {_missing}'
assert _P(TRACKER_PATH).parent.exists(), f'Tracker directory not found: {TRACKER_PATH}'
print('✓ All model and data files found')
print(f'  XGB:      {XGB_MODEL_PATH}')
print(f'  Ensemble: {ENS_MODEL_PATH}')
print(f'  LightGBM: {LGBM_MODEL_PATH}')
del _P, _missing

✓ All model and data files found
  XGB:      c:\Users\josep\Desktop\random_stuff\BettingEdgeContinued\betting\models\xgboost_prod_model.pkl
  Ensemble: c:\Users\josep\Desktop\random_stuff\BettingEdgeContinued\betting\models\ensemble_prod_model.pkl
  LightGBM: c:\Users\josep\Desktop\random_stuff\BettingEdgeContinued\betting\models\lgbm_prod_model.pkl


## Load XGBoost Model

Loads the production XGBoost sklearn pipeline (`xgboost_prod_model.pkl`). The pipeline has two transformer slots inside its `preprocessor` step — one for categorical columns (`roof`, `surface`) and one for all numeric features — followed by an `XGBRegressor`. `model_features` is derived by introspecting those transformer slots so it always stays in sync with whatever the pkl was trained on.

In [6]:
xgb_res      = joblib.load(XGB_MODEL_PATH)
pipeline     = xgb_res['pipeline']
pre          = pipeline.named_steps['preprocessor']
_known_cats  = set(FinalCfg().categorical_cols)
cat_cols, num_cols = None, None
for _, _, _cols in pre.transformers_:
    if set(_cols) & _known_cats:
        cat_cols = list(_cols)
    else:
        num_cols = list(_cols)
assert cat_cols is not None and num_cols is not None, "Could not identify categorical/numeric transformer slots"
model_features = cat_cols + num_cols
print(f"XGBoost loaded — {len(model_features)} features")

XGBoost loaded — 77 features


In [7]:
assert pipeline is not None, 'XGBoost pipeline not loaded'
assert hasattr(pipeline, 'predict'), 'pipeline missing predict method'
assert 'preprocessor' in pipeline.named_steps, 'pipeline missing preprocessor step'
assert len(model_features) > 0, 'model_features is empty'
assert 'roof'        in model_features, 'roof missing from model_features'
assert 'surface'     in model_features, 'surface missing from model_features'
assert 'spread_line' in model_features, 'spread_line missing from model_features'
print(f'✓ XGBoost: {len(model_features)} features')
print(f'  First 6: {model_features[:6]}')

✓ XGBoost: 77 features
  First 6: ['roof', 'surface', 'spread_line', 'away_rest', 'home_rest', 'total_line']


## Load Ensemble Model — fixed75

Loads the primary **edge-setting** model: a fixed 0.75 XGBoost / 0.25 Ridge blend (`ensemble_prod_model.pkl`). This model's `ens_model_edge` drives the HIGH/MEDIUM/PASS confidence threshold and determines game ranking. Ridge is stored inside the same pkl — no separate file is needed.

In [ ]:
# ── Load Ensemble (fixed75) ───────────────────────────────────────────────────
ens_pkg        = joblib.load(ENS_MODEL_PATH)
ens_xgb        = ens_pkg["xgb_model"]
ens_ridge      = ens_pkg["ridge_model"]
ens_scaler     = ens_pkg["scaler"]
ens_xgb_weight = ens_pkg["xgb_weight"]
ens_feat_cols  = ens_pkg["feature_cols"]
ens_enc        = ens_pkg["roof_surface_encoder"]
print(f"Ensemble loaded — XGB weight={ens_xgb_weight}")

In [ ]:
assert ens_xgb_weight == 0.75, f'Expected XGB weight 0.75, got {ens_xgb_weight}'
assert hasattr(ens_xgb,   'predict'), 'ens_xgb missing predict'
assert hasattr(ens_ridge, 'predict'), 'ens_ridge missing predict'
assert ens_scaler is not None, 'ens_scaler not loaded'
assert ens_enc    is not None, 'ens_enc not loaded'
assert len(ens_feat_cols) > 0, 'ens_feat_cols empty'
print(f'✓ Ensemble (fixed75): XGB weight={ens_xgb_weight} | {len(ens_feat_cols)} features')

## Load LightGBM Model

Loads the third direction voter (`lgbm_prod_model.pkl`). LightGBM uses leaf-wise tree growth, making it a genuinely independent signal from XGBoost (which grows level-wise). All three voters — XGBoost standalone, Ridge, and LightGBM — must agree on direction for a HIGH or MEDIUM confidence pick.

In [ ]:
# ── Load LightGBM voter ──────────────────────────────────────────────────────
lgbm_pkg       = joblib.load(LGBM_MODEL_PATH)
lgbm_model     = lgbm_pkg["model"]
lgbm_feat_cols = lgbm_pkg["feature_cols"]
print(f"LightGBM loaded — {len(lgbm_feat_cols)} features")

In [ ]:
assert lgbm_model is not None, 'LightGBM model not loaded'
assert hasattr(lgbm_model, 'predict'), 'lgbm_model missing predict'
assert len(lgbm_feat_cols) > 0, 'lgbm_feat_cols empty'
print(f'✓ LightGBM: {len(lgbm_feat_cols)} features')
print(f'  Feature counts — XGB: {len(model_features)}, '
      f'Ensemble: {len(ens_feat_cols)}, LightGBM: {len(lgbm_feat_cols)}')

## Static Data — AllPro Roster & Team Map

Loads the All-Pro CSV (updated manually each January). `TEAM_MAP` normalises historical franchise abbreviations so records from 1997 onward join cleanly to modern schedule data (e.g. `STL`→`LA`, `ARZ`→`ARI`). `2TM` rows — players who split a season between two teams — are dropped because their split-year credit is already counted under each individual team row.

In [ ]:
# ── Load static data ──────────────────────────────────────────────────────────
allpro_df = pd.read_csv(ALLPRO_CSV_PATH)
allpro_df = allpro_df[allpro_df["Team"] != "2TM"].copy()

TEAM_MAP = {
    "STL": "LA", "LAR": "LA", "OAK": "LV", "LVR": "LV",
    "SD": "LAC", "SDG": "LAC", "NWE": "NE", "KAN": "KC",
    "GNB": "GB", "NOR": "NO", "TAM": "TB", "SFO": "SF",
    # Pre-2002 / alternate abbreviations present in historical AllPro CSV
    "ARZ": "ARI", "BLT": "BAL", "CLV": "CLE", "HST": "HOU", "JAC": "JAX",
}
allpro_df["Team"] = allpro_df["Team"].replace(TEAM_MAP)

In [ ]:
assert len(allpro_df) > 0, 'allpro_df is empty'
assert '2TM' not in allpro_df['Team'].values, '2TM rows not filtered'
for _k in ['STL', 'OAK', 'SD', 'ARZ', 'BLT', 'CLV', 'HST', 'JAC']:
    assert _k in TEAM_MAP, f'TEAM_MAP missing historical key: {_k}'
for _old in ['ARZ', 'BLT', 'STL']:
    assert _old not in allpro_df['Team'].values, f'{_old} not remapped'
print(f'✓ AllPro: {len(allpro_df)} rows | seasons {allpro_df["Year"].min()}–{allpro_df["Year"].max()}')
print(f'✓ TEAM_MAP: {len(TEAM_MAP)} entries | all historical abbreviations present')
del _k, _old

## Helper: `_norm_name`

Normalises a player name string for fuzzy joining between the weekly injury report and the All-Pro CSV. The two sources use inconsistent casing, suffixes (Jr./Sr./II/III), accented characters, and punctuation — raw string equality would miss roughly 15% of injured All-Pros. Stripping all of these before joining recovers those matches.

In [ ]:
def _norm_name(s):
    if not isinstance(s, str): return ''
    s = ''.join(c for c in _ud.normalize('NFD', s) if _ud.category(c) != 'Mn')
    s = s.lower().strip()
    s = _re.sub(r'\s+(jr\.?|sr\.?|ii|iii|iv|v)\s*$', '', s)
    s = _re.sub(r"[\'.\-]", '', s)
    return _re.sub(r'\s+', ' ', s).strip()

In [ ]:
assert _norm_name('Patrick Mahomes Jr.') == 'patrick mahomes', 'Jr. not stripped'
assert _norm_name('Odell Beckham II')     == 'odell beckham',   'II not stripped'
assert _norm_name('John Smith III')       == 'john smith',      'III not stripped'
assert _norm_name("D'Andre Swift")        == 'dandre swift',    'apostrophe not removed'
assert _norm_name(None)                   == '',                'None not handled'
assert _norm_name('  Tom Brady  ')        == 'tom brady',       'whitespace not stripped'
assert _norm_name('LAMAR JACKSON')        == 'lamar jackson',   'not lowercased'
print('✓ _norm_name: Jr./II/III stripping, apostrophes, whitespace, None — all pass')

## Helper: `get_week_info`

Detects the next unplayed week (first week with `result IS NULL`) and the immediately preceding completed week. Accepts an optional `schedule_df` parameter to reuse a DataFrame already in memory — avoids a redundant `nfl.load_schedules` API call when the schedule was loaded earlier in the pipeline.

In [ ]:
# ── Helper: detect current/previous week ─────────────────────────────────────
def get_week_info(season, schedule_df=None):
    if schedule_df is None:
        raw      = nfl.load_schedules([season])
        schedule = raw.to_pandas() if hasattr(raw, 'to_pandas') else pd.DataFrame(raw)
    else:
        schedule = schedule_df
    reg      = schedule[(schedule['season'] == season) & (schedule['game_type'] == 'REG')]
    future   = reg[reg['result'].isna()]
    done     = reg[reg['result'].notna()]
    if future.empty:
        return None, int(done['week'].max())
    upcoming_week = int(future['week'].min())
    prev_week     = upcoming_week - 1 if upcoming_week > 1 else None
    return upcoming_week, prev_week

In [ ]:
_s = pd.DataFrame([
    {'season':2025,'week':w,'game_type':'REG',
     'result':7.0 if w<5 else None,
     'home_score':24 if w<5 else None,
     'away_score':17 if w<5 else None}
    for w in range(1, 8)
])
_up, _pv = get_week_info(2025, schedule_df=_s)
assert _up == 5 and _pv == 4, f'Mid-season: expected (5,4), got ({_up},{_pv})'

_done = _s.copy(); _done['result'] = 7.0
_up2, _ = get_week_info(2025, schedule_df=_done)
assert _up2 is None, 'Complete season should return upcoming=None'

_w1 = _s.copy(); _w1['result'] = None
_up3, _pv3 = get_week_info(2025, schedule_df=_w1)
assert _up3 == 1 and _pv3 is None, f'Week-1: expected (1,None), got ({_up3},{_pv3})'

print('✓ get_week_info: mid-season, complete-season, week-1 — all pass')
del _s, _done, _w1, _up, _pv, _up2, _up3, _pv3

## Feature engineering — loaded from `features.ipynb`

The 85-feature engineering pipeline (Groups 1–10) is the **single source of
truth** in [features.ipynb](features.ipynb). This notebook loads those
definitions via `%run features.ipynb`, so the names `build_features`,
`build_numeric_features`, the per-group `_build_*` helpers, `FEATURE_COLS_85`,
`PROD_FEATURES_35`, `TEAM_MAP`, and `norm_name` all land in this kernel's
namespace.

We set `RUN_TESTS = False` before the `%run` so the synthetic-data tests
in `features.ipynb` (which take ~1 second) don't run on every prediction
kickoff — production correctness is validated by the inline test cells
that follow here, against the same live nflreadpy data the pipeline uses.


In [ ]:
# Load all feature-engineering helpers from the shared notebook.
# We use json.load + exec rather than %run because:
#   (a) %run is brittle across runners (papermill / nbclient / VSCode) and CWDs.
#   (b) The json-exec pattern works identically in any environment that can
#       open a file — including the Streamlit container, if app.py ever needs
#       these helpers.
# RUN_TESTS=False skips the synthetic tests inside features.ipynb; the inline
# test cells in *this* notebook (against live nflreadpy data) cover the same surface.
import json as _json
RUN_TESTS = False
_FEATURES_NB = _DIR / "features.ipynb"
assert _FEATURES_NB.exists(), f"features.ipynb not found at {_FEATURES_NB}"
with open(_FEATURES_NB, encoding="utf-8") as _f:
    _features_nb = _json.load(_f)
for _cell in _features_nb["cells"]:
    if _cell["cell_type"] != "code":
        continue
    _src = _cell["source"]
    if isinstance(_src, list):
        _src = "".join(_src)
    exec(_src, globals())
del _f, _features_nb, _cell, _src, _FEATURES_NB
print(f"Loaded feature engineering from features.ipynb — "
      f"{len(FEATURE_COLS_85)} canonical features, {len(PROD_FEATURES_35)} in production subset.")


In [ ]:
# Minimal synthetic inputs — Groups 7 and 9 call nfl APIs internally but fall back
# gracefully on failure, so no mocking is needed.
_hist = [
    {'season':2025,'week':w,'game_id':f'2025_W{w}_1','game_type':'REG',
     'home_team':'KC','away_team':'BUF','spread_line':-3.0,'total_line':47.0,
     'result':7.0,'home_score':24,'away_score':17,'roof':'dome','surface':'turf',
     'home_rest':7,'away_rest':7,'div_game':0,
     'home_coach':'Andy Reid','away_coach':'Sean McDermott',
     'home_qb_name':'P.Mahomes','away_qb_name':'J.Allen','gameday':'2025-10-05'}
    for w in range(1, 5)
]
_up_row = {**_hist[0], 'week':5, 'game_id':'2025_W5_1',
           'result':None, 'home_score':None, 'away_score':None}
_fsched = pd.DataFrame(_hist + [_up_row])

# 25 plays per team per game gives enough attempts for passer-rating filter (>=100)
_pbp_rows = [
    {'season':s,'week':w,'game_id':f'{s}_W{w}_1','posteam':t,'defteam':o,
     'play_id':p,'play_type':'pass','epa':0.1,'yards_gained':6,'sack':0,
     'interception':0,'fumble_lost':0,'down':1,'first_down':1,
     'pass_attempt':1,'complete_pass':1,'passing_yards':7,'pass_touchdown':0,
     'passer_player_name':'P.Mahomes' if t=='KC' else 'J.Allen'}
    for s in [2024,2025]
    for w in (range(1,19) if s==2024 else range(1,5))
    for t,o in [('KC','BUF'),('BUF','KC')]
    for p in range(25)
]
_pbp = pd.DataFrame(_pbp_rows)
_allpro = pd.DataFrame({'Year':[2024,2023],'Team':['KC','BUF'],
                         'Player':['P1','P2'],'Side':['offense','defense']})
_coach_hist = _fsched[_fsched['result'].notna()].copy()

_res = build_features(
    target_week=5, target_season=2025,
    full_schedule=_fsched, pbp_rp=_pbp, allpro_df=_allpro,
    week_margin_lkp=None, coach_hist_df=_coach_hist
)
assert _res is not None,                 'build_features returned None'
assert isinstance(_res, pd.DataFrame),   'build_features did not return a DataFrame'
assert len(_res) == 1,                   f'Expected 1 upcoming row, got {len(_res)}'
for _c in ['home_rolling_win_pct','home_coach_win_pct_prior',
           'home_coach_win_pct_roll3','sos_diff','cover_rate_diff',
           'home_pr_prev_year','home_cpae_prev_year','home_time_to_throw_prev_year',
           'diff_pr_prev_year','diff_cpae_prev_year','diff_time_to_throw_prev_year']:
    assert _c in _res.columns,           f'Expected column missing: {_c}'
    assert not pd.isna(_res[_c].iloc[0]),f'Column {_c} is NaN'
print(f'✓ build_features: {len(_res)} game row, {len(_res.columns)} columns, key features present')
print(f'  home_rolling_win_pct={_res["home_rolling_win_pct"].iloc[0]:.2f}  '
      f'coach_prior={_res["home_coach_win_pct_prior"].iloc[0]:.3f}  '
      f'coach_roll3={_res["home_coach_win_pct_roll3"].iloc[0]:.3f}')
del _hist, _up_row, _fsched, _pbp_rows, _pbp, _allpro, _coach_hist, _res, _c

## Helper: `build_numeric_features`

Loaded by the `%run` in cell 28. See [features.ipynb](features.ipynb) for the
implementation. The test cell below re-validates it post-%run against the
encoder we'll use at inference time.


In [ ]:
# build_numeric_features was loaded by cell 28's %run features.ipynb.
# This cell exists only to keep the cell-numbering aligned with CLAUDE.md's table;
# the test cell below exercises the function post-%run.


In [ ]:
from sklearn.preprocessing import OrdinalEncoder as _OE
_enc = _OE(handle_unknown='use_encoded_value', unknown_value=-1)
_enc.fit([['dome','turf'],['outdoors','grass'],['retractable','turf']])
_feats = ['roof','surface','is_playoff','spread_line']

# Known categories -> (1,4) float32 array
_df_ok = pd.DataFrame({'roof':['dome'],'surface':['turf'],'is_playoff':[0],'spread_line':[-3.0]})
_X = build_numeric_features(_df_ok, _feats, _enc)
assert isinstance(_X, np.ndarray),              'Expected numpy array'
assert _X.shape == (1, 4),                       f'Expected (1,4), got {_X.shape}'
assert np.issubdtype(_X.dtype, np.floating),     'Expected float dtype'

# Unknown category -> must not raise
_df_unk = pd.DataFrame({'roof':['open_air_xyz'],'surface':['sod'],'is_playoff':[0],'spread_line':[2.5]})
try:
    build_numeric_features(_df_unk, _feats, _enc); _ok = True
except Exception: _ok = False
assert _ok, 'Unknown roof/surface raised unexpectedly'

# NaN input -> must not raise
_df_nan = pd.DataFrame({'roof':[None],'surface':[None],'is_playoff':[0],'spread_line':[-1.0]})
try:
    build_numeric_features(_df_nan, _feats, _enc); _nan_ok = True
except Exception: _nan_ok = False
assert _nan_ok, 'NaN roof/surface raised unexpectedly'

print('✓ build_numeric_features: known categories, unknown fallback, NaN input — all pass')
del _OE, _enc, _feats, _df_ok, _X, _df_unk, _ok, _df_nan, _nan_ok

## Helper: `run_predictions`

Runs all four models on the feature matrix and assembles the results table. Each model is given exactly the columns it was trained on (via the `feature_cols` list stored inside each pkl — 35 numerics for the Ensemble and LightGBM; 37 columns including raw `roof` / `surface` for the standalone XGBoost pipeline). `ens_model_edge` (Ensemble fixed75) is the primary sort key. `consensus_tier` is:

- **HIGH** — XGBoost standalone, Ridge, and LightGBM all agree on direction **and** `abs(ens_model_edge) ≥ 3 pts`
- **MEDIUM** — all three agree **and** `≥ 1 pt`
- **PASS** — any disagreement, or edge < 1 pt

The Ensemble itself is the edge-setter, not a voter — XGBoost standalone, Ridge, and LightGBM are the three direction voters.

In [ ]:
# ── Run predictions ───────────────────────────────────────────────────────────
def run_predictions(target_week, target_season, full_schedule, pbp_rp, allpro_df, week_margin_lkp=None, coach_hist_df=None):
    print(f"Building features for season {target_season} week {target_week}...")
    upcoming = build_features(target_week, target_season, full_schedule, pbp_rp, allpro_df,
                              week_margin_lkp=week_margin_lkp, coach_hist_df=coach_hist_df,
                              required_features=list(dict.fromkeys(model_features + ens_feat_cols + lgbm_feat_cols)))
    if upcoming is None or upcoming.empty:
        print("No games found.")
        return None

    # ── XGBoost (prod) ───────────────────────────────────────────────────────
    X     = upcoming[model_features].copy()
    preds = pipeline.predict(X)

    # ── Ensemble (fixed75: 0.75 XGB + 0.25 Ridge) ────────────────────────────
    X_ens_raw = build_numeric_features(upcoming, ens_feat_cols, ens_enc)
    X_ens_sc  = ens_scaler.transform(X_ens_raw)
    ens_preds = (ens_xgb_weight * ens_xgb.predict(X_ens_raw)
                 + (1 - ens_xgb_weight) * ens_ridge.predict(X_ens_sc))

    # ── Ridge (extracted from ensemble pkg) ─────────────────────────────────────
    ridge_preds = ens_ridge.predict(X_ens_sc)

    # ── LightGBM (independent voter) ─────────────────────────────────────────
    X_lgbm    = build_numeric_features(upcoming, lgbm_feat_cols, ens_enc)
    lgbm_preds = lgbm_model.predict(X_lgbm)

    results = upcoming[['game_id','home_team','away_team','gameday','spread_line']].copy()
    spread  = results['spread_line']

    results['predicted_margin']     = preds.round(1)
    results['model_edge']           = (results['predicted_margin'] - spread).round(1)
    results['ens_predicted_margin']   = ens_preds.round(1)
    results['ens_model_edge']         = (results['ens_predicted_margin'] - spread).round(1)
    results['ridge_predicted_margin'] = ridge_preds.round(1)
    results['ridge_model_edge']       = (results['ridge_predicted_margin'] - spread).round(1)
    results['lgbm_predicted_margin']  = lgbm_preds.round(1)
    results['lgbm_model_edge']        = (results['lgbm_predicted_margin'] - spread).round(1)

    def side(edge, home, away):
        if edge > 0:  return f"HOME ({home})"
        if edge < 0:  return f"AWAY ({away})"
        return "PASS"

    results['recommendation']     = results.apply(lambda r: side(r['model_edge'],     r['home_team'], r['away_team']), axis=1)
    results['ens_recommendation']   = results.apply(lambda r: side(r['ens_model_edge'],   r['home_team'], r['away_team']), axis=1)
    results['ridge_recommendation'] = results.apply(lambda r: side(r['ridge_model_edge'], r['home_team'], r['away_team']), axis=1)
    results['lgbm_recommendation']  = results.apply(lambda r: side(r['lgbm_model_edge'],  r['home_team'], r['away_team']), axis=1)

    # Consensus tier: HIGH = XGB (standalone)/Ridge/LightGBM all agree direction + abs(ens_model_edge) >= 3pt
    def consensus_tier(row):
        sides = [row['recommendation'], row['ridge_recommendation'], row['lgbm_recommendation']]
        agree = all(s != 'PASS' for s in sides) and len(set(sides)) == 1
        edge  = abs(row['ens_model_edge'])
        if agree and edge >= 3: return 'HIGH'
        if agree and edge >= 1: return 'MEDIUM'
        return 'PASS'
    results['consensus_tier'] = results.apply(consensus_tier, axis=1)

    results = results.sort_values('ens_model_edge', key=abs, ascending=False)
    display_cols = ['home_team','away_team','spread_line','ens_model_edge','ridge_model_edge','model_edge','lgbm_model_edge','consensus_tier']
    print(results[display_cols].to_string(index=False))
    return results

## Helper: `update_results`

After games are played, fetches actual scores from `nflreadpy` and fills `actual_margin`, `home_covered`, and per-model `*_correct` columns in the tracker CSV. Push outcomes (margin exactly equals spread) are stored as `NaN` — they don't count toward ATS accuracy in either direction. The ATS denominator uses `notna().sum()` to exclude push games.

In [ ]:
# ── Update results ────────────────────────────────────────────────────────────
def update_results(season, week):
    if not os.path.exists(TRACKER_PATH):
        print("No tracker found — skipping results update")
        return
    print(f"Updating results for season {season} week {week}...")
    tracker = pd.read_csv(TRACKER_PATH)
    raw     = nfl.load_schedules([season])
    sched   = raw.to_pandas() if hasattr(raw, 'to_pandas') else pd.DataFrame(raw)

    # Pull result AND individual scores
    actual  = sched[(sched['season'] == season) & (sched['week'] == week)][
        ['game_id', 'result', 'home_score', 'away_score']
    ].rename(columns={'result': 'actual_margin'})

    if actual['actual_margin'].isna().all():
        print(f"Results not yet available for week {week} — skipping")
        return
    mask    = (tracker['season'] == season) & (tracker['week'] == week)
    indices = tracker[mask].index
    if len(indices) == 0:
        print(f"No predictions found for week {week} — skipping")
        return
    rows = tracker.loc[indices].copy()
    rows = rows.merge(actual, on='game_id', how='left', suffixes=('_old', '_new'))
    for _col in ['actual_margin', 'home_score', 'away_score']:
        if f'{_col}_new' in rows.columns:
            rows[_col] = rows[f'{_col}_new']
    rows = rows.drop(columns=['actual_margin_old', 'actual_margin_new', 'home_score_old', 'home_score_new', 'away_score_old', 'away_score_new'], errors='ignore')
    def _home_covered(margin, spread):
        if pd.isna(margin) or pd.isna(spread):
            return float('nan')
        if margin == spread:  # push — no ATS result
            return float('nan')
        return float(margin > spread)
    rows['home_covered'] = rows.apply(
        lambda r: _home_covered(r['actual_margin'], r['spread_line']), axis=1
    )
    def _score(edge, covered):
        if pd.isna(edge) or pd.isna(covered) or edge == 0:
            return float('nan')
        return int((edge > 0) == (covered == 1))

    rows['model_correct']     = rows.apply(lambda r: _score(r['model_edge'],     r['home_covered']), axis=1)
    if 'ens_model_edge' in rows.columns:
        rows['ens_model_correct'] = rows.apply(lambda r: _score(r['ens_model_edge'],   r['home_covered']), axis=1)
    if 'ridge_model_edge' in rows.columns:
        rows['ridge_model_correct'] = rows.apply(lambda r: _score(r['ridge_model_edge'], r['home_covered']), axis=1)
    if 'lgbm_model_edge' in rows.columns:
        rows['lgbm_model_correct'] = rows.apply(lambda r: _score(r['lgbm_model_edge'],  r['home_covered']), axis=1)

    tracker.loc[indices, 'actual_margin'] = rows['actual_margin'].values
    tracker.loc[indices, 'home_covered']  = rows['home_covered'].values
    tracker.loc[indices, 'model_correct'] = rows['model_correct'].values
    tracker.loc[indices, 'home_score']    = rows['home_score'].values
    tracker.loc[indices, 'away_score']    = rows['away_score'].values
    for col in ['ens_model_correct', 'ridge_model_correct', 'lgbm_model_correct']:
        if col in rows.columns:
            tracker.loc[indices, col] = rows[col].values
    tracker.to_csv(TRACKER_PATH, index=False)

    ens_col = 'ens_model_correct' if 'ens_model_correct' in rows.columns else 'model_correct'
    correct = int(rows[ens_col].sum())
    total   = int(rows[ens_col].notna().sum())
    if total > 0:
        print(f"✅ Week {week} ATS (Ensemble): {correct}/{total} ({correct/total*100:.1f}%)")
    else:
        print(f"✅ Week {week}: results updated (no model-predicted games)")

In [ ]:
import math as _m

# _home_covered: push -> NaN; cover -> 1.0; no-cover -> 0.0
def _hc(margin, spread):
    if pd.isna(margin) or pd.isna(spread): return float('nan')
    if margin == spread: return float('nan')  # push
    return float(margin > spread)

assert _hc(7.0,  3.0) == 1.0,                   'Win by more than spread -> cover'
assert _hc(1.0,  3.0) == 0.0,                   'Win by less than spread -> no cover'
assert _m.isnan(_hc(-3.0, -3.0)),                'Exact push -> NaN'
assert _m.isnan(_hc(float('nan'), 3.0)),          'Unknown margin -> NaN'

# _score: zero edge -> NaN (model had no opinion)
def _sc(edge, covered):
    if pd.isna(edge) or pd.isna(covered) or edge == 0: return float('nan')
    return int((edge > 0) == (covered == 1))

assert _sc( 3.0, 1.0) == 1,                     'Positive edge + cover -> correct'
assert _sc( 3.0, 0.0) == 0,                     'Positive edge + no cover -> wrong'
assert _sc(-2.0, 0.0) == 1,                     'Negative edge + no cover -> correct'
assert _m.isnan(_sc(0.0, 1.0)),                   'Zero edge -> NaN'
assert _m.isnan(_sc(3.0, float('nan'))),           'Unknown covered -> NaN'

# ATS % denominator must exclude push (NaN) rows
_mc = pd.Series([1, 0, float('nan'), 1, float('nan')])
assert abs(_mc.sum() / _mc.notna().sum() - 2/3) < 1e-9, 'notna denominator should give 2/3'

print('✓ update_results: push->NaN, cover logic, zero-edge NaN, notna denominator — all pass')
del _m, _hc, _sc, _mc

## Helper: `log_predictions`

Appends (or refreshes) a week's predictions in `betting/predictions_tracker.csv`. On a thursday/sunday refresh, it removes the old week entry but copies back any already-filled result columns (`actual_margin`, `home_covered`, `model_correct`, individual scores) so a re-run after games are played does not wipe the results.

In [ ]:
# ── Log predictions ───────────────────────────────────────────────────────────
def log_predictions(results_df, season, week, mode):
    extra = [c for c in ['ens_predicted_margin','ens_model_edge','ens_recommendation',
                          'ridge_predicted_margin','ridge_model_edge','ridge_recommendation',
                          'lgbm_predicted_margin','lgbm_model_edge','lgbm_recommendation',
                          'consensus_tier'] if c in results_df.columns]
    log = results_df[['game_id','home_team','away_team','gameday','spread_line',
                       'predicted_margin','model_edge','recommendation'] + extra].copy()
    log['season']        = season
    log['week']          = week
    log['mode']          = mode
    log['logged_at']     = datetime.now().strftime('%Y-%m-%d %H:%M')
    log['actual_margin'] = None
    log['home_covered']  = None
    log['model_correct'] = None
    log['home_score']    = None
    log['away_score']    = None
    if os.path.exists(TRACKER_PATH):
        tracker = pd.read_csv(TRACKER_PATH)
        mask    = (tracker['season'] == season) & (tracker['week'] == week)
        if mask.any():
            print(f"Replacing existing week {week} predictions ({mode} refresh)...")
            _res_cols = ['actual_margin', 'home_covered', 'model_correct', 'home_score', 'away_score',
                         'ens_model_correct', 'ridge_model_correct', 'lgbm_model_correct']
            old_results = tracker.loc[mask, ['game_id'] + [c for c in _res_cols if c in tracker.columns]].copy()
            tracker = tracker[~mask]
            log = log.merge(
                old_results.rename(columns={c: f'_old_{c}' for c in _res_cols}),
                on='game_id', how='left'
            )
            for col in _res_cols:
                old_col = f'_old_{col}'
                if old_col in log.columns:
                    log[col] = log[old_col]
                    log = log.drop(columns=[old_col])
        updated = pd.concat([tracker, log], ignore_index=True)
        updated.to_csv(TRACKER_PATH, index=False)
    else:
        log.to_csv(TRACKER_PATH, index=False)
    print(f"✅ Week {week} predictions saved ({mode} — {len(log)} games)")

In [ ]:
import tempfile as _tf, os as _os, io as _io, sys as _sys

_tmpf = _tf.NamedTemporaryFile(suffix='.csv', delete=False, mode='w')
_tmpf.close()
# Tracker already has week 1 with results filled
pd.DataFrame({
    'game_id':['2025_01_KC_BUF'],'season':[2025],'week':[1],
    'home_team':['KC'],'away_team':['BUF'],'gameday':['2025-09-07'],
    'spread_line':[-3.0],'predicted_margin':[4.0],'model_edge':[1.0],
    'recommendation':['HOME'],'mode':['tuesday'],'logged_at':['2025-09-02 10:00'],
    'actual_margin':[7.0],'home_covered':[1.0],'model_correct':[1],
    'home_score':[27],'away_score':[20],'ens_model_correct':[1],
    'ridge_model_correct':[1],'lgbm_model_correct':[1],
}).to_csv(_tmpf.name, index=False)

_orig = TRACKER_PATH
TRACKER_PATH = _tmpf.name   # redirect writes to temp file
_res = pd.DataFrame({
    'game_id':['2025_01_KC_BUF'],'home_team':['KC'],'away_team':['BUF'],
    'gameday':['2025-09-07'],'spread_line':[-3.0],
    'predicted_margin':[4.2],'model_edge':[1.2],'recommendation':['HOME'],
    'ens_predicted_margin':[4.2],'ens_model_edge':[1.2],'ens_recommendation':['HOME'],
    'ridge_predicted_margin':[3.8],'ridge_model_edge':[0.8],'ridge_recommendation':['HOME'],
    'lgbm_predicted_margin':[4.5],'lgbm_model_edge':[1.5],'lgbm_recommendation':['HOME'],
    'consensus_tier':['HIGH'],
})
_buf = _io.StringIO(); _saved = _sys.stdout; _sys.stdout = _buf
log_predictions(_res, season=2025, week=1, mode='thursday')
_sys.stdout = _saved

_row = pd.read_csv(_tmpf.name)
_row = _row[_row['game_id']=='2025_01_KC_BUF'].iloc[0]
assert _row['actual_margin']    == 7.0, 'actual_margin not preserved on refresh'
assert _row['home_covered']     == 1.0, 'home_covered not preserved'
assert _row['model_correct']    == 1,   'model_correct not preserved'
assert _row['home_score']       == 27,  'home_score not preserved'
assert _row['ens_model_correct']== 1,   'ens_model_correct not preserved'

TRACKER_PATH = _orig
_os.unlink(_tmpf.name)
print('✓ log_predictions: thursday refresh preserves actual_margin, scores, model_correct — all pass')
del _tf, _os, _io, _sys, _tmpf, _orig, _res, _buf, _saved, _row

## Run Pipeline

Auto-detects the season from the current date (month ≥ September = new season). Loads all schedules (1999–present) in a **single** `nfl.load_schedules` call and derives `full_schedule`, `coach_hist_df`, and `week_margin_lkp` from the same DataFrame — no duplicate API fetches.

| Mode | Trigger | Behaviour |
|------|---------|----------|
| `tuesday` | Tue 9am ET | Updates previous week results → runs new predictions → logs to tracker |
| `thursday` | Thu 9pm ET | Refreshes predictions with latest injury data → overwrites week entry |
| `sunday` | Sun 9am ET | Final predictions before kickoff → overwrites week entry |
| `backfill` | Manual | Runs predictions for `TARGET_WEEK` → logs → immediately fills results |

In [ ]:
# Auto-detect season from current date if not overridden
if TARGET_SEASON is None:
    _now = datetime.now()
    TARGET_SEASON = _now.year if _now.month >= 9 else _now.year - 1
    print(f"Auto-detected TARGET_SEASON={TARGET_SEASON}")

# Load all schedules in one call (1999–present) — provides full_schedule,
# coach history, and week-margin lookup without duplicate API fetches
print("Loading schedules (1999–present, single pass)...")
_raw_all   = nfl.load_schedules(list(range(1999, TARGET_SEASON + 1)))
_all_sched = _raw_all.to_pandas() if hasattr(_raw_all, 'to_pandas') else pd.DataFrame(_raw_all)
_all_sched['season'] = _all_sched['season'].astype(int)
_all_sched['week']   = _all_sched['week'].astype(int)

full_schedule = _all_sched[_all_sched['season'] == TARGET_SEASON].copy().reset_index(drop=True)
coach_hist_df = _all_sched[_all_sched['result'].notna()].copy()
_hist_df_lkp  = _all_sched[
    (_all_sched['season'].between(2014, 2022)) &
    (_all_sched['game_type'] == 'REG') & _all_sched['result'].notna()
]
week_margin_lkp = _hist_df_lkp.groupby('week')['result'].apply(lambda x: x.abs().mean())
print(f"Schedules loaded: {len(full_schedule)} current-season games | "
      f"{len(coach_hist_df)} completed (coach) | {len(week_margin_lkp)} week-margin keys")

if TARGET_WEEK is None:
    TARGET_WEEK, PREV_WEEK = get_week_info(TARGET_SEASON, schedule_df=full_schedule)
    if TARGET_WEEK is None:
        raise ValueError("Season is over — no predictions to run. See you in September!")
else:
    TARGET_WEEK = int(TARGET_WEEK)
    PREV_WEEK   = (TARGET_WEEK - 1) if TARGET_WEEK > 1 else None

print(f"Upcoming week: {TARGET_WEEK} | Previous week: {PREV_WEEK}")

print("Loading PBP data (this takes ~60s)...")
raw_pbp = nfl.load_pbp([TARGET_SEASON, TARGET_SEASON - 1])
pbp     = raw_pbp.to_pandas() if hasattr(raw_pbp, 'to_pandas') else pd.DataFrame(raw_pbp)
pbp_rp  = pbp[
    pbp['play_type'].isin(['run','pass']) &
    pbp['posteam'].notna() &
    pbp['defteam'].notna()
].copy()
print(f"PBP loaded: {pbp_rp.shape} | Seasons: {sorted(pbp_rp['season'].unique())}")

if MODE == 'tuesday':
    if PREV_WEEK:
        update_results(TARGET_SEASON, PREV_WEEK)
    if TARGET_WEEK:
        results = run_predictions(TARGET_WEEK, TARGET_SEASON, full_schedule, pbp_rp, allpro_df, week_margin_lkp=week_margin_lkp, coach_hist_df=coach_hist_df)
        if results is not None:
            log_predictions(results, TARGET_SEASON, TARGET_WEEK, mode='tuesday')

# thursday and sunday run the same prediction logic;
# the only difference is when in the week they execute (injury data freshness).
elif MODE == 'thursday':
    if TARGET_WEEK:
        results = run_predictions(TARGET_WEEK, TARGET_SEASON, full_schedule, pbp_rp, allpro_df, week_margin_lkp=week_margin_lkp, coach_hist_df=coach_hist_df)
        if results is not None:
            log_predictions(results, TARGET_SEASON, TARGET_WEEK, mode='thursday')

elif MODE == 'sunday':
    if TARGET_WEEK:
        results = run_predictions(TARGET_WEEK, TARGET_SEASON, full_schedule, pbp_rp, allpro_df, week_margin_lkp=week_margin_lkp, coach_hist_df=coach_hist_df)
        if results is not None:
            log_predictions(results, TARGET_SEASON, TARGET_WEEK, mode='sunday')

elif MODE == 'backfill':
    if TARGET_WEEK:
        results = run_predictions(TARGET_WEEK, TARGET_SEASON, full_schedule, pbp_rp, allpro_df, week_margin_lkp=week_margin_lkp, coach_hist_df=coach_hist_df)
        if results is not None:
            log_predictions(results, TARGET_SEASON, TARGET_WEEK, mode='backfill')
            update_results(TARGET_SEASON, TARGET_WEEK)

else:
    print(f"Unknown mode: {MODE}. Use tuesday, thursday, sunday, or backfill.")
